# Proyecto Integrador de Machine Learning - Pre-Entrega

## Dataset: Titanic - Supervivencia de pasajeros

### 1. Selección del dataset

**Origen**: El dataset Titanic está disponible en Seaborn y también en archivos CSV públicos (por ejemplo, de Kaggle). En este notebook lo cargaremos directamente desde la URL de Seaborn.

**Descripción del problema**: Predecir si un pasajero sobrevivió al hundimiento del Titanic (variable objetivo `survived`) a partir de características como clase, sexo, edad, tarifa, etc. Es un problema de **clasificación binaria**.

**Justificación**: Es un dataset clásico, con tamaño pequeño (891 registros), mezcla de variables numéricas y categóricas, valores faltantes y algunos outliers. Permite aplicar limpieza, codificación, escalado y selección de variables sin necesidad de recursos computacionales altos, ideal para esta pre-entrega.

---

## 2. Análisis exploratorio inicial (EDA)

Cargamos librerías, dataset y realizamos un primer vistazo.

In [ ]:
!pip install seaborn pandas numpy matplotlib scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

titanic = sns.load_dataset('titanic')

print("Primeras 5 filas:")
titanic.head()

In [ ]:
print("Información del dataset:")
titanic.info()

print("\nEstadísticas descriptivas:")
titanic.describe()

print("\nVariables categóricas:")
titanic.describe(include=['object', 'category'])

In [ ]:
print("Valores nulos:")
titanic.isnull().sum()

plt.figure(figsize=(10, 4))
sns.heatmap(titanic.isnull(), cbar=False, yticklabels=False)
plt.title("Mapa de valores faltantes")
plt.show()

In [ ]:
sns.countplot(x='survived', data=titanic)
plt.title("Distribución de Supervivencia")
plt.show()

print("Proporción:", titanic['survived'].mean())

sns.histplot(titanic['age'].dropna(), bins=30, kde=True)
plt.title("Edad")
plt.show()

sns.boxplot(x='pclass', y='fare', data=titanic)
plt.title("Tarifa por clase")
plt.show()

In [ ]:
numeric_cols = titanic.select_dtypes(include=[np.number]).columns

plt.figure(figsize=(8, 6))
sns.heatmap(titanic[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlación")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='sex', hue='survived', data=titanic, ax=axes[0])
axes[0].set_title("Sexo")

sns.countplot(x='pclass', hue='survived', data=titanic, ax=axes[1])
axes[1].set_title("Clase")

plt.show()

---

## 3. Limpieza de datos

In [ ]:
df = titanic.copy()

print("Nulos antes:")
print(df.isnull().sum())

df.drop(columns=['deck', 'alive', 'who', 'adult_male', 'embark_town'], inplace=True, errors='ignore')

df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
df['fare'].fillna(df['fare'].median(), inplace=True)

print("\nNulos después:")
print(df.isnull().sum())

print("Duplicados:", df.duplicated().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(y=df['age'], ax=axes[0])
sns.boxplot(y=df['fare'], ax=axes[1])

plt.show()

cap = df['fare'].quantile(0.99)
df['fare_capped'] = np.where(df['fare'] > cap, cap, df['fare'])

---

## 4. Transformaciones

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le_sex = LabelEncoder()
df['sex_encoded'] = le_sex.fit_transform(df['sex'])

le_embarked = LabelEncoder()
df['embarked_encoded'] = le_embarked.fit_transform(df['embarked'])

cols_to_scale = ['age', 'fare_capped', 'sibsp', 'parch']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

df['family_size'] = df['sibsp'] + df['parch'] + 1

df.drop(columns=['sex', 'embarked', 'fare', 'sibsp', 'parch'], inplace=True)

df.head()

In [ ]:
df.isnull().sum()

---

## 5. Selección de variables

In [ ]:
y = df['survived']
X = df.drop(columns=['survived'])

correlations = X.corrwith(y).sort_values(ascending=False)
print(correlations)

selected_features = correlations[abs(correlations) > 0.1].index.tolist()
print("Seleccionadas:", selected_features)

X_selected = X[selected_features]

In [ ]:
X_selected.head()
y.head()

---

## 6. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

---

## Resumen

Dataset listo para modelado 